# Simplest BERT-style Transformer on BabiStories
Encoder-only masked language model. Compact and based on the previous notebook style.

In [68]:
import torch,random,math,re,json
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import Counter
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cfg={
    "d_model":128,
    "d_ff":512,
    "num_heads":4,
    "num_layers":4,
    "max_vocab":12000,
    "seq_len":128,
    "batch_size":4,
    "lr":3e-4,
    "epochs":100,
    "mask_prob":0.15,
    "dropout":0.05,
    "weight_decay":0.1,
    "use_wq":False,
    "use_aq":False,
    "w_bits":8,
    "a_bits":8,
    "use_ws":False,
    "ws_ratio":0.0,
    "attn_top_k":None,
    "norm":"pre"
}

cuda


## Data

##### Load BabiStories Dataset

In [69]:
def clean_text(t):
    t=t.replace("<MASK>"," MASKTOKEN ").replace("<mask>"," MASKTOKEN ")
    t=t.lower()
    t=re.sub(r"([.,!?])", r" \1 ", t)
    t=re.sub(r"[^a-z0-9\s\.\,\!\?']", " ", t)
    t=t.replace("masktoken"," <MASK> ")
    t=re.sub(r"\s+", " ", t).strip()
    return t

def read_texts_from_file(p):
    texts=[]
    if p.suffix.lower()==".txt":
        x=p.read_text(encoding="utf-8",errors="ignore")
        for t in re.split(r"\n\s*\n|\n",x):
            t=clean_text(t)
            if len(t.split())>20:texts.append(t)
    elif p.suffix.lower()==".jsonl":
        for line in p.open(encoding="utf-8",errors="ignore"):
            if line.strip():
                d=json.loads(line)
                t=d.get("text") or d.get("story") or d.get("content") or ""
                t=clean_text(t)
                if len(t.split())>20:texts.append(t)
    elif p.suffix.lower()==".json":
        d=json.loads(p.read_text(encoding="utf-8",errors="ignore"))
        if isinstance(d,list):
            for e in d:
                t=e.get("text") or e.get("story") or e.get("content") or ""
                t=clean_text(t)
                if len(t.split())>20:texts.append(t)
    return texts

def load_babistories(folder="BabiStories/data/extracted"):
    folder=Path(folder);texts=[]
    if not folder.exists():raise FileNotFoundError(folder)
    for p in folder.rglob("*"):
        if p.suffix.lower() in [".txt",".jsonl",".json"]:
            texts+=read_texts_from_file(p)
    if len(texts)==0:raise ValueError("No .txt/.json/.jsonl stories found")
    return texts

texts=load_babistories()
random.seed(42);random.shuffle(texts)
print("texts:",len(texts))
print(texts[0][:300])
print("mask test:",clean_text("the girl went to the <MASK> to play ."))

texts: 2221736
at the edge of a sparkling river , a playful otter discovered a lost baby bird named soleil . soleil was worried and cold , separated from her mom and siblings . the otter felt bad and wanted to help . he knew of a warm , cozy nest close by , but he wasn't sure if soleil would want to leave the plac
mask test: the girl went to the <MASK> to play .


In [ ]:
def clean_text(t):
    t=t.replace("<MASK>"," MASKTOKEN ").replace("<mask>"," MASKTOKEN ")
    t=t.lower()
    t=re.sub(r"([.,!?])", r" \1 ", t)
    t=re.sub(r"[^a-z0-9\s\.\,\!\?']", " ", t)
    t=t.replace("masktoken"," <MASK> ")
    t=re.sub(r"\s+", " ", t).strip()
    return t

texts=load_babistories()
texts=[clean_text(t) for t in texts]
texts=[t for t in texts if len(t.split())>30]
random.seed(42)
random.shuffle(texts)

def build_context(texts,cfg):
    sp=["<PAD>","<MASK>","<UNK>"]
    c=Counter()
    for t in texts:
        c.update(t.split())
    vocab=sp+[w for w,n in c.most_common(cfg["max_vocab"]-len(sp)) if n>=3 and w not in sp]
    stoi={w:i for i,w in enumerate(vocab)}
    itos={i:w for w,i in stoi.items()}
    return {
        "stoi":stoi,
        "itos":itos,
        "vocab":vocab,
        "pad":stoi["<PAD>"],
        "mask":stoi["<MASK>"],
        "unk":stoi["<UNK>"],
        "seq_len":cfg["seq_len"]
    }

def encode(text,ctx,max_len):
    ids=[ctx["stoi"].get(w,ctx["unk"]) for w in text.split()]
    ids=ids[:max_len]
    ids=ids+[ctx["pad"]]*(max_len-len(ids))
    return ids

def make_examples(texts,ctx,max_examples=30000,max_unk_ratio=0.02):
    ex=[]
    for t in texts:
        ids=[ctx["stoi"].get(w,ctx["unk"]) for w in t.split()]
        for i in range(0,max(1,len(ids)-ctx["seq_len"]+1),ctx["seq_len"]//2):
            s=ids[i:i+ctx["seq_len"]]
            if len(s)<16:continue
            unk=sum(1 for x in s if x==ctx["unk"])
            if unk/len(s)>max_unk_ratio:continue
            s=s+[ctx["pad"]]*(ctx["seq_len"]-len(s))
            ex.append(s)
            if len(ex)>=max_examples:return ex
    return ex

def make_mlm_batch(data,ctx,batch_size,device):
    b=random.choices(data,k=batch_size) if len(data)<batch_size else random.sample(data,batch_size)
    x=torch.tensor(b,dtype=torch.long,device=device)
    y=torch.full_like(x,-100)
    valid=x.ne(ctx["pad"])&x.ne(ctx["unk"])&x.ne(ctx["mask"])
    pos=(torch.rand(x.shape,device=device)<cfg["mask_prob"])&valid
    y[pos]=x[pos]
    r=torch.rand(x.shape,device=device)
    mask_pos=pos&(r<0.8)
    rand_pos=pos&(r>=0.8)&(r<0.9)
    x[mask_pos]=ctx["mask"]
    random_words=torch.randint(3,len(ctx["vocab"]),x.shape,device=device)
    x[rand_pos]=random_words[rand_pos]
    return x,y,x.eq(ctx["pad"])

ctx=build_context(texts,cfg)
examples=make_examples(texts,ctx,max_examples=30000,max_unk_ratio=0.02)
random.shuffle(examples)
n=int(0.8*len(examples))
v=int(0.1*len(examples))
train_data=examples[:n]
valid_data=examples[n:n+v]
test_data=examples[n+v:]

print("train:",len(train_data),"valid:",len(valid_data),"test:",len(test_data),"vocab:",len(ctx["vocab"]))
print("mask id:",ctx["mask"],"mask token id:",ctx["stoi"].get("<MASK>"))
print("encode mask test:",encode(clean_text("the girl went to the <MASK> to play ."),ctx,ctx["seq_len"])[:10])

train: 24000 valid: 3000 test: 3000 vocab: 12000
mask id: 1 mask token id: 1
encode mask test: [5, 96, 81, 8, 5, 1, 8, 53, 3, 0]


## Quantization and sparsity hooks

In [71]:
def qste(x,bits,use):
    if not use:return x
    qmax=2**(bits-1)-1;s=x.abs().max().clamp(min=1e-8)/qmax
    y=(x/s).round().clamp(-qmax,qmax)*s
    return x+(y-x).detach()
def sparsify(w,ratio,use):
    if not use or ratio<=0:return w
    th=torch.quantile(w.abs().flatten(),ratio)
    return w*(w.abs()>=th)
def qw(w,cfg):return qste(sparsify(w,cfg["ws_ratio"],cfg["use_ws"]),cfg["w_bits"],cfg["use_wq"])
def qa(x,cfg):return qste(x,cfg["a_bits"],cfg["use_aq"])
def lin(x,l,cfg):return F.linear(qa(x,cfg),qw(l.weight,cfg),l.bias)

## Model

In [72]:
class MHA(nn.Module):
    def __init__(self,d_model,num_heads,cfg):
        super().__init__();self.h=num_heads;self.dh=d_model//num_heads;self.cfg=cfg
        self.q=nn.Linear(d_model,d_model);self.k=nn.Linear(d_model,d_model);self.v=nn.Linear(d_model,d_model);self.o=nn.Linear(d_model,d_model)
    def forward(self,x,key_pad=None):
        B,T,D=x.shape
        Q=lin(x,self.q,self.cfg).view(B,T,self.h,self.dh).transpose(1,2)
        K=lin(x,self.k,self.cfg).view(B,T,self.h,self.dh).transpose(1,2)
        V=lin(x,self.v,self.cfg).view(B,T,self.h,self.dh).transpose(1,2)
        S=Q@K.transpose(-2,-1)/math.sqrt(self.dh)
        if key_pad is not None:S=S.masked_fill(key_pad[:,None,None,:],-1e9)
        if self.cfg["attn_top_k"] is not None:
            k=min(self.cfg["attn_top_k"],T);th=S.topk(k,dim=-1).values[...,-1,None];S=S.masked_fill(S<th,-1e9)
        A=F.softmax(S,dim=-1);O=(A@V).transpose(1,2).contiguous().view(B,T,D)
        return lin(O,self.o,self.cfg),A

class FFN(nn.Module):
    def __init__(self,d_model,d_ff,cfg):
        super().__init__();self.l1=nn.Linear(d_model,d_ff);self.l2=nn.Linear(d_ff,d_model);self.cfg=cfg
    def forward(self,x):
        return lin(F.relu(lin(x,self.l1,self.cfg)),self.l2,self.cfg)

class Block(nn.Module):
    def __init__(self,d_model,d_ff,num_heads,cfg):
        super().__init__();self.a=MHA(d_model,num_heads,cfg);self.f=FFN(d_model,d_ff,cfg);self.n1=nn.LayerNorm(d_model);self.n2=nn.LayerNorm(d_model);self.cfg=cfg
    def forward(self,x,pad):
        if self.cfg["norm"]=="post":a,A=self.a(x,pad);x=self.n1(x+a);x=self.n2(x+self.f(x))
        else:n=self.n1(x);a,A=self.a(n,pad);x=x+a;x=x+self.f(self.n2(x))
        return x,A

class BERTMLM(nn.Module):
    def __init__(self,vocab_size,cfg,ctx):
        super().__init__();d=cfg["d_model"];self.cfg=cfg;self.ctx=ctx
        self.tok=nn.Embedding(vocab_size,d,padding_idx=ctx["pad"]);self.pos=nn.Embedding(ctx["seq_len"],d)
        self.blocks=nn.ModuleList([Block(d,cfg["d_ff"],cfg["num_heads"],cfg) for _ in range(cfg["num_layers"])])
        self.n=nn.LayerNorm(d);self.out=nn.Linear(d,vocab_size)
    def forward(self,x,pad):
        p=torch.arange(x.size(1),device=x.device)[None,:];h=self.tok(x)+self.pos(p);A=[]
        for b in self.blocks:h,a=b(h,pad);A.append(a)
        return lin(self.n(h),self.out,self.cfg),A

## Train

In [73]:
model=BERTMLM(len(ctx["vocab"]),cfg,ctx).to(device)
opt=torch.optim.AdamW(model.parameters(),lr=cfg["lr"],weight_decay=cfg["weight_decay"])

def acc_mlm(logits,y):
    m=y.ne(-100)
    if m.sum().item()==0:return 0.0
    return (logits.argmax(-1).eq(y)&m).sum().item()/m.sum().item()

def step(data,train=True):
    model.train(train);x,y,pad=make_mlm_batch(data,ctx,cfg["batch_size"],device)
    with torch.set_grad_enabled(train):
        logits,A=model(x,pad)
        loss=F.cross_entropy(logits.reshape(-1,logits.size(-1)),y.reshape(-1),ignore_index=-100)
        if train:
            opt.zero_grad();loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step()
    return loss.item(),acc_mlm(logits,y)

def evaluate(data,n=20):
    loss=0;acc=0
    for _ in range(n):
        l,a=step(data,False);loss+=l;acc+=a
    return loss/n,acc/n

def train_epochs(epochs=20,train_steps=100,valid_steps=20,patience=15):
    history={"train_loss":[],"train_acc":[],"val_loss":[],"val_acc":[]};best=1e9;state=None
    for e in range(1,epochs+1):
        tl=ta=0
        for _ in range(train_steps):
            l,a=step(train_data,True);tl+=l;ta+=a
        vl,va=evaluate(valid_data,valid_steps);tl/=train_steps;ta/=train_steps
        history["train_loss"].append(tl);history["train_acc"].append(ta);history["val_loss"].append(vl);history["val_acc"].append(va)
        if vl<best:
            best=vl;bad=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            bad+=1
        print(e,"train_loss",tl,"train_acc",ta,"val_loss",vl,"val_acc",va)
        if bad>=patience:
            print("early stop at epoch",e,"best_val_loss",best)
            break
    return best_state,history

best_state,history=train_epochs(epochs=cfg["epochs"],train_steps=100,valid_steps=50,patience=15)
model.load_state_dict(best_state)

1 train_loss 7.4001120805740355 train_acc 0.07094179479545887 val_loss 6.310902585983277 val_acc 0.08967651652844733
2 train_loss 6.219244513511658 train_acc 0.08238246103630205 val_loss 6.1920842266082765 val_acc 0.08726101221362041
3 train_loss 6.109838976860046 train_acc 0.09170547184986783 val_loss 6.191556615829468 val_acc 0.08762246316238827
4 train_loss 6.090140399932861 train_acc 0.0954338673161094 val_loss 6.0529015922546385 val_acc 0.08683093661800555
5 train_loss 6.084963569641113 train_acc 0.09598203092404707 val_loss 6.052257022857666 val_acc 0.10058851903495784
6 train_loss 6.032323813438415 train_acc 0.10276673871069875 val_loss 5.963045768737793 val_acc 0.10032044210774389
7 train_loss 6.028774852752686 train_acc 0.0998738613056032 val_loss 5.886337823867798 val_acc 0.10653064321529344
8 train_loss 5.940297265052795 train_acc 0.10339801241455286 val_loss 5.955617151260376 val_acc 0.11640289532391111
9 train_loss 5.949381041526794 train_acc 0.10878608950498048 val_loss 5

<All keys matched successfully>

## Fill mask

In [74]:
def fill_mask(text,topk=10):
    model.eval()
    text=clean_text(text)
    ids=encode(text,ctx,ctx["seq_len"])
    pos=[i for i,v in enumerate(ids) if v==ctx["mask"]]

    print("cleaned:",text)
    print("mask id:",ctx["mask"])
    print("mask positions:",pos)

    if len(pos)==0:
        return []

    x=torch.tensor([ids],dtype=torch.long,device=device)
    pad=x.eq(ctx["pad"])

    with torch.no_grad():
        logits,A=model(x,pad)

    out=[]
    for p in pos:
        logit=logits[0,p].clone()
        logit[ctx["pad"]]=-1e9
        logit[ctx["mask"]]=-1e9
        logit[ctx["unk"]]=-1e9
        probs=F.softmax(logit,dim=-1)
        vals,idx=probs.topk(topk)
        out.append([(ctx["itos"][int(i)],float(v)) for v,i in zip(vals,idx)])
    return out

print(fill_mask("the girl went to the <MASK> to play .",10))
print(fill_mask("the boy ate a <MASK> because he was hungry .",10))
print(fill_mask("she opened the <MASK> and went inside .",10))
print(fill_mask("he was tired so he went to <MASK> .",10))

cleaned: the girl went to the <MASK> to play .
mask id: 1
mask positions: [5]
[[('the', 0.12730726599693298), (',', 0.06376001238822937), ('.', 0.03557252883911133), ('a', 0.02893218770623207), ('and', 0.02293901890516281), ('to', 0.013911579735577106), ('!', 0.013020854443311691), ('i', 0.01062301080673933), ('was', 0.009177440777420998), ('of', 0.009010152891278267)]]
cleaned: the boy ate a <MASK> because he was hungry .
mask id: 1
mask positions: [4]
[[(',', 0.10623786598443985), ('a', 0.09259990602731705), ('.', 0.06979259848594666), ('the', 0.05591128021478653), ('he', 0.02754334546625614), ('to', 0.01992369256913662), ('in', 0.018635712563991547), ('his', 0.017898235470056534), ('of', 0.017866840586066246), ('and', 0.017594486474990845)]]
cleaned: she opened the <MASK> and went inside .
mask id: 1
mask positions: [3]
[[('the', 0.07174164056777954), ('.', 0.06891635805368423), (',', 0.06393039971590042), ('she', 0.03376710042357445), ('a', 0.032034073024988174), ('and', 0.02865205

In [75]:
def overfit_one_batch(steps=500):
    model.train()
    x,y,pad=make_mlm_batch(train_data,ctx,cfg["batch_size"],device)

    for i in range(steps):
        logits,A=model(x,pad)
        loss=F.cross_entropy(
            logits.reshape(-1,logits.size(-1)),
            y.reshape(-1),
            ignore_index=-100
        )
        opt.zero_grad()
        loss.backward()
        opt.step()

        if i%50==0:
            acc=acc_mlm(logits,y)
            print(i,loss.item(),acc)


overfit_one_batch()

0 5.821596145629883 0.1686746987951807
50 0.11964569240808487 1.0
100 0.02509147860109806 1.0
150 0.01720423810184002 1.0
200 0.01339853834360838 1.0
250 0.011066803708672523 1.0
300 0.009474369697272778 1.0
350 0.008311284705996513 1.0
400 0.0074210395105183125 1.0
450 0.0067156353034079075 1.0


## Later switches

In [76]:
# cfg["use_wq"]=True;cfg["use_aq"]=True
# cfg["use_ws"]=True;cfg["ws_ratio"]=0.5
# cfg["attn_top_k"]=8